    # Pregunta 2 — Mapeo de inundaciones con Sentinel-1, paso a paso

    Durante una inundación hay que saber **dónde está el agua** en las horas
    siguientes. Las nubes que acompañan a la tormenta dejan ciego al satélite
    óptico justo cuando más se lo necesita; el radar de apertura sintética
    atraviesa la nube y ve de noche. La tarea es segmentar agua píxel a píxel
    sobre imágenes SAR de Sentinel-1, con una **U-Net propia**.


> **Cómo usar este cuaderno.** Las celdas de código son **seguras de correr**:
> inspeccionan los datos y los resultados que ya están en disco, pero
> **ninguna reentrena nada**. Los comandos que sí entrenan están escritos como
> texto, no como celdas, para que no se disparen por accidente al hacer
> "ejecutar todo".
>
> Si una celda dice que falta un archivo, es que ese paso todavía no corrió:
> el README del paso correspondiente tiene el comando exacto.

## Lo que hay que tener antes de empezar

| qué | cómo |
|---|---|
| Sen1Floods11 (7.5 GB) | `python pregunta2/paso1_datos/descargar.py` |
| Las estadísticas | `python pregunta2/paso1_datos/estadisticas.py` |
| El modelo entrenado | `python pregunta2/paso3_entrenamiento/entrenar.py` |

Los datos quedan accesibles desde `pregunta2/datos/` gracias a
`alinear_datos.py`, que crea enlaces de directorio: **no duplica un solo
byte**, así que `pregunta2/datos/sen1floods11/x.tif` y
`data/sen1floods11/x.tif` son el mismo archivo en disco.

In [ ]:
import json
import pathlib
import sys

# La raiz del proyecto se busca hacia arriba, asi el cuaderno funciona tanto
# si se abre desde la carpeta de la pregunta como desde la raiz.
# Raiz del proyecto en la maquina local. Orden: la variable REDES_RAIZ, la
# carpeta actual o alguna superior, y por ultimo la ruta fija de esta PC. Asi
# funciona en Jupyter/VS Code y en Colab conectado a un entorno de ejecucion
# local, aunque Jupyter se haya lanzado desde otra carpeta.
import os
RAIZ_LOCAL = pathlib.Path("D:/Backend/redesneuronales")
RAIZ = pathlib.Path(os.environ.get("REDES_RAIZ", pathlib.Path.cwd()))
while not (RAIZ / "pregunta1").is_dir() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
if not (RAIZ / "pregunta1").is_dir():
    RAIZ = RAIZ_LOCAL
if not (RAIZ / "pregunta1").is_dir():
    raise FileNotFoundError(
        f"No encuentro el proyecto (busque desde {pathlib.Path.cwd()} y en "
        f"{RAIZ_LOCAL}). En Colab: Conectar -> 'Conectar a un entorno de "
        "ejecucion local', o define REDES_RAIZ con la ruta del proyecto.")
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
print("raiz del proyecto:", RAIZ)


def leer(ruta):
    """Lee un JSON de resultados; avisa con claridad si todavia no existe.

    Acepta una ruta relativa a la raiz o una absoluta (la que da rutas.py).
    """
    p = pathlib.Path(ruta)
    if not p.is_absolute():
        p = RAIZ / p
    if not p.is_file():
        print(f"  (falta {ruta}: ese paso todavia no corrio)")
        return None
    return json.loads(p.read_text(encoding="utf-8"))

In [ ]:
# Que hay en disco, antes de cualquier otra cosa.
from pregunta2.rutas import V11, describir
print(describir())
base = V11 / "data/flood_events"
for nombre, sub in (("SAR dorado", "HandLabeled/S1Hand"),
                    ("etiqueta dorada", "HandLabeled/LabelHand"),
                    ("linea base Otsu", "HandLabeled/S1OtsuLabelHand"),
                    ("agua permanente", "HandLabeled/JRCWaterHand"),
                    ("SAR debil", "WeaklyLabeled/S1Weak")):
    d = base / sub
    n = len(list(d.glob("*.tif"))) if d.is_dir() else 0
    print(f"  {nombre:<18} {n:>5} chips")

# Paso 1 — Los datos, y por qué medirlos antes de normalizar

Hay una tentación fuerte de normalizar con las constantes de ImageNet
(`mean=(0.485, 0.456, 0.406)`) porque están en todos los cuadernos. Acá
**no tienen ningún sentido**: la entrada no son tres canales de color
entre 0 y 1, son **dos canales de retrodispersión en decibeles**, que son
negativos y se mueven en un rango de decenas.

Así que se miden. Y se miden sobre el **split de entrenamiento oficial**,
nunca sobre validación ni prueba: calcular la normalización con datos de
prueba es fuga, chica pero fuga.

In [ ]:
est = leer("pregunta2/paso1_datos/estadisticas.json")
if est:
    print(f"medido sobre {est['chips']} chips del split de ENTRENAMIENTO\n")
    print(f"{'banda':<6} {'media':>9} {'desvio':>9} {'min crudo':>11}")
    print("-" * 38)
    for i, b in enumerate(est["bandas"]):
        print(f"{b:<6} {est['media'][i]:>9.3f} {est['desvio'][i]:>9.3f} "
              f"{est['min'][i]:>11.2f}")
    print("\nEse minimo de -79 dB es esencialmente log(0): pixeles donde la")
    print("retrodispersion dio cero. Un solo valor asi estira la escala y")
    print("aplasta todo lo demas, por eso se recorta por percentiles.")

## Las etiquetas tienen **tres** valores, no dos

Esta es la decisión que más condiciona todo lo que sigue.

In [ ]:
if est:
    e = {int(k): v for k, v in est["etiquetas"].items()}
    total = sum(e.values())
    nombres = {1: "agua", 0: "no agua", -1: "NO SE PUDO ETIQUETAR"}
    for k in (1, 0, -1):
        print(f"  {k:>2}  {nombres[k]:<22} {e[k]:>12,}  {100*e[k]/total:>6.2f}%")
    obs = e[0] + e[1]
    print(f"\n  prevalencia de agua entre las observadas: {100*e[1]/obs:.2f}%")
    print(f"\n  -> {100*e[-1]/total:.1f}% de los pixeles NO se pueden supervisar.")
    print("     Tratarlos como 'no agua' meteria supervision falsa en uno de")
    print("     cada siete pixeles: la perdida los tiene que ENMASCARAR.")
    print(f"\n  -> con {100*e[1]/obs:.1f}% de agua, la exactitud por pixel es")
    print("     inutil: predecir 'nunca hay agua' ya acierta el 90.5%.")
    print("     Por eso la metrica del enunciado es IoU.")

### La pérdida enmascarada

Es el mismo problema que en la Pregunta 1 con las etiquetas inciertas de
CheXpert, y la misma solución: una máscara que entra **en el divisor**.

El detalle que rompe si se hace mal: un lote donde no haya **ningún**
píxel observado tiene que devolver cero **con gradiente**. Devolver un
`torch.tensor(0.)` suelto corta el grafo y `backward` falla.

In [ ]:
import torch
from pregunta2.paso1_datos.sen1floods11 import perdida_enmascarada

logits = torch.randn(2, 1, 64, 64, requires_grad=True)
objetivo = torch.zeros(2, 1, 64, 64)
mascara = torch.zeros(2, 1, 64, 64)      # ni un solo pixel observado

p = perdida_enmascarada(logits, objetivo, mascara)
p.backward()
print(f"perdida: {p.item()}")
print(f"gradiente existe: {logits.grad is not None}")
print(f"sin NaN: {torch.isfinite(logits.grad).all().item()}")

# Paso 2 — La U-Net propia (entregable 1)

El enunciado pide implementarla **sin copiar código de bibliotecas de
segmentación**. Lo único de terceros acá son las primitivas de PyTorch.

La estructura sigue las notas de clase: codificador de dos convoluciones
3×3 + ReLU por nivel con max-pooling 2×2 duplicando canales, cuello de
botella, y decodificador que **concatena** —no suma— con el mapa del
codificador del mismo nivel.

In [ ]:
from pregunta2.paso2_modelo.unet import construir_unet

print(f"{'variante':<12} {'parametros':>11}  canales")
print("-" * 52)
for v in ("base", "angosta", "profunda", "somera", "bilineal", "sin_norm"):
    m = construir_unet(v)
    print(f"{v:<12} {m.parametros_millones:>9.1f} M  {m.config.canales}")

m = construir_unet("base", canales_entrada=2, clases=1)
x = torch.zeros(2, 2, 512, 512)
with torch.no_grad():
    y = m(x)
print(f"\nentrada {tuple(x.shape)} -> salida {tuple(y.shape)}")
print("  mismo tamano espacial: la mascara se alinea pixel a pixel con la")
print("  grilla georreferenciada del GeoTIFF, sin reproyectar nada.")

### Tres apartamientos del artículo de 2015, y por qué

1. **Relleno "same"** en vez de convoluciones sin relleno. El artículo
   recorta el mapa del codificador antes de concatenar; las notas de clase
   dicen que las implementaciones modernas usan relleno "simplificando la
   implementación sin cambiar la idea central". Acá además importa por el
   dominio: con relleno la salida mide **exactamente** lo mismo que la
   entrada y se alinea con el GeoTIFF.
2. **BatchNorm.** El artículo entrena con lotes de tamaño 1 y momentum
   0.99 justamente para compensar su ausencia. Con 512×512 en una GPU
   moderna entran lotes de 8–16.
3. **Profundidad configurable**, porque el entregable 2(b) la admite como
   factor de ablación.

# Paso 3 — Entrenar y evaluar (entregables 2 y 3)

Dos decisiones del entrenamiento que valen la pena:

- **Se elige la época por IoU de validación, no por pérdida.** La pérdida
  mezcla BCE y Dice y mejora aunque la IoU no se mueva.
- **Bolivia se mira una sola vez, al final.** Es la región retenida:
  usarla para elegir hiperparámetros la convertiría en otra validación.

In [ ]:
r = leer("pregunta2/paso3_entrenamiento/corridas/a_solo_dorados/resultados.json")
if r:
    print(f"variante {r['variante']} | {r['parametros_millones']} M parametros")
    print(f"mejor epoca {r['mejor_epoca']} de {r['epocas']} | {r['minutos']:.1f} min\n")
    print(f"{'conjunto':<24} {'IoU agreg':>10} {'IoU/chip':>10} {'F1':>8} "
          f"{'prec':>8} {'exh':>8}")
    print("-" * 72)
    for n in ("valid", "test", "bolivia"):
        m = r["resultados"][n]
        etiqueta = n + (" (RETENIDA)" if n == "bolivia" else "")
        print(f"{etiqueta:<24} {m['iou_agua']:>10.4f} "
              f"{m['iou_promedio_por_chip']:>10.4f} {m['f1']:>8.4f} "
              f"{m['precision']:>8.4f} {m['exhaustividad']:>8.4f}")

### Por qué dos IoU y no una

La **agregada** suma los conteos de todos los chips y divide una vez: pesa
más los chips con mucha agua. La **promedio por chip** pesa todos igual, y
por eso sale mucho más baja —hay chips casi secos donde la IoU es
inestable—. El enunciado pide la segunda, porque es la que reportan los
autores de Sen1Floods11. Reportar solo una dejaría la comparación con el
paper a medias.

In [ ]:
if r:
    c = r["resultados"]["test"]["confusion"]
    tot = sum(c.values())
    print(f"MATRIZ DE CONFUSION sobre prueba ({tot:,} pixeles observados)\n")
    print(f"{'':>16} {'pred agua':>14} {'pred no agua':>14}")
    print(f"{'real agua':>16} {c['vp']:>14,} {c['fn']:>14,}")
    print(f"{'real no agua':>16} {c['fp']:>14,} {c['vn']:>14,}")
    om = 100*c['fn']/(c['vp']+c['fn'])
    co = 100*c['fp']/(c['vp']+c['fp'])
    print(f"\n  omision  (agua real que se perdio) : {om:.2f}%")
    print(f"  comision (agua predicha que no era): {co:.2f}%")
    print("\n  Los dos errores estan casi balanceados, que es lo que uno")
    print("  quiere de un umbral en 0.5. Para respuesta a emergencias")
    print("  convendria moverlo hacia menos omision: perder agua real es")
    print("  mas caro que una falsa alarma.")

# Paso 4 — El resultado principal (entregable 4)

Acá aparece lo más interesante de toda la pregunta, y es un resultado
incómodo.

In [ ]:
c = leer("pregunta2/paso4_generalizacion/comparacion.json")
if c:
    print(f"{'conjunto':<12} {'U-Net':>8} {'Otsu':>8} {'delta':>9} "
          f"{'IC95%':>22} {'sig':>5}")
    print("-" * 68)
    for n, d in c["conjuntos"].items():
        sig = "SI" if d["significativa"] else "no"
        print(f"{n:<12} {d['iou_unet']:>8.4f} {d['iou_otsu']:>8.4f} "
              f"{d['delta']:>+9.4f}  [{d['ic95'][0]:+.4f}, "
              f"{d['ic95'][1]:+.4f}] {sig:>5}")
    ca = c["caida_unet_test_a_bolivia"]
    print(f"\ncaida de la U-Net de prueba a Bolivia: {ca['delta']:+.4f}")
    print(f"  IC95% [{ca['ic95'][0]:+.4f}, {ca['ic95'][1]:+.4f}] -> "
          f"{'significativa' if ca['significativa'] else 'NO se distingue del ruido'}")

### Cómo leer eso, y cómo **no** leerlo

Las dos diferencias contra Otsu son significativas y **el signo se da
vuelta**: lo aprendido le gana al método clásico dentro de la distribución
y se lo pierde fuera.

Pero la caída de la **propia U-Net** entre prueba y Bolivia no se distingue
del ruido. Entonces la lectura correcta **no** es "la U-Net se derrumba en
Bolivia", sino:

> **Otsu mejora mucho en Bolivia (0.546 → 0.697) y la U-Net no lo acompaña.**

Bolivia es una región fácil de umbralizar —agua abierta, contraste
limpio— y el modelo aprendido no aprovecha eso. Decirlo de la otra forma
sería sobreinterpretar 15 chips.

### Por qué bootstrap **pareado**, y sobre **chips**

Los dos métodos se evalúan sobre los mismos chips, así que remuestrear los
chips y mirar la *diferencia* cancela el ruido del conjunto. Con intervalos
sueltos, dos intervalos anchos que se solapan no dicen nada aunque la
diferencia sea estable.

Y se remuestrean **chips enteros, nunca píxeles**: los píxeles de un mismo
chip están fuertemente correlacionados —el agua viene en manchas, no
salpicada— y tratarlos como independientes daría intervalos artificialmente
angostos.

## La ablación (entregable 2)

Siete corridas, misma partición y misma semilla, cambiando una cosa por vez.

In [ ]:
a = leer("pregunta2/paso3_entrenamiento/ablacion.json")
if a:
    import numpy as np
    print(f"{'corrida':<18} {'factor':<13} {'IoU prueba':>11} {'IoU Bolivia':>12}")
    print("-" * 58)
    for f in sorted(a, key=lambda x: -x["iou_test"]):
        print(f"{f['corrida']:<18} {f['factor']:<13} "
              f"{f['iou_test']:>11.4f} {f['iou_bolivia']:>12.4f}")
    print(f"{'Otsu (sin entrenar)':<18} {'-':<13} "
          f"{a[0]['otsu_test']:>11.4f} {a[0]['otsu_bolivia']:>12.4f}")

    t = np.array([x["iou_test"] for x in a])
    b = np.array([x["iou_bolivia"] for x in a])
    rt, rb = np.argsort(np.argsort(t)), np.argsort(np.argsort(b))
    print(f"\ncorrelacion de rangos entre las dos columnas: "
          f"{np.corrcoef(rt, rb)[0,1]:+.3f}")
    print("  negativa y fuerte: lo que mejora dentro empeora fuera.")
    print(f"\nninguna configuracion supera a Otsu en Bolivia; la mejor queda")
    print(f"  {a[0]['otsu_bolivia'] - b.max():+.4f} por debajo.")

### El hallazgo del factor (a), que es el que el enunciado exige

Agregar etiquetas débiles **empeora dentro de distribución y mejora
fuera**. Tiene explicación directa: las débiles salen de una umbralización
de Otsu, así que entrenar con ellas empuja al modelo hacia el
comportamiento del método clásico —peor donde ya conoce el terreno, más
robusto en una región nueva—.

### Un cuidado que es fácil pasar por alto

Los 4 384 chips débiles **incluyen 224 de Bolivia**. Sumarlos "porque son
del otro conjunto" le mostraría al modelo la región sobre la que después
se mide la generalización, y la brecha no significaría nada.

# Paso 5 — La región peruana (entregable 4)

El enunciado nombra Lima explícitamente y pide **construir** la región de
prueba combinando dos fuentes independientes. Bolivia es el respaldo, no
el camino principal.

In [ ]:
from pregunta2.rutas import LIMA
li = leer(LIMA / "chips.json")
if li:
    ch = li["chips"]
    con = sum(1 for x in ch if x["pixeles_agua"] > 0)
    agua = sum(x["pixeles_agua"] for x in ch)
    obs = sum(x["pixeles_observados"] for x in ch)
    print(f"{len(ch)} chips de {li['lado']}x{li['lado']} | "
          f"paso {li['paso_grados']:.3e} grados (~10 m)")
    print(f"  con agua: {con} | secos: {len(ch)-con}")
    print(f"  agua entre lo observado: {100*agua/obs:.2f}%")
    print(f"\nfuente: {li['fuente_etiquetas']}")
    print(f"huella : {li['fuente_huella']}")
    print(f"\nLIMITE DECLARADO:\n  {li['limite_declarado']}")

### Dónde se puede afirmar algo y dónde no

La capa `Footprints` del geodatabase trae las 6 huellas de imagen con su
fecha. Solo dos son post-evento: **WorldView-2 del 18 y del 21 de marzo de
2017**. Fuera de esa unión **nadie miró**, así que no es "no inundado": es
*desconocido*, y va como `-1`.

Escribir 0 ahí inventaría supervisión negativa sobre media Lima y haría ver
al modelo mucho mejor de lo que es.

### Y acá aparece la segunda hipótesis del entregable 4

El agua es **~1%** de los píxeles observados en Lima, contra **9.51%** en
el entrenamiento. Lima es casi diez veces más seca que lo que el modelo
vio entrenando.

# Resumen de los seis entregables

| # | qué pide | estado |
|---|---|---|
| 1 | U-Net propia, sin copiar bibliotecas de segmentación | **hecho** |
| 2 | Ablación de ≥2 decisiones | **hecho** — 7 corridas |
| 3 | IoU, F1, precisión, recall + matriz de confusión | **hecho** |
| 4 | Generalización al Perú | **parcial** — falta la imagen SAR de Lima |
| 5 | ≥3 fallos característicos del SAR | pendiente |
| 6 | Bono: fusión con Sentinel-2 | no empezado |

# Las tres lecciones de esta pregunta

**1. Medir antes de normalizar.** Copiar las constantes de ImageNet sobre
decibeles habría dejado la entrada descentrada y mal escalada, sin dar
ningún error.

**2. Un método sin entrenar puede ganarle a uno entrenado, y hay que
medirlo.** Si no se hubiera comparado contra Otsu, el 0.6096 de Bolivia
habría pasado por un resultado razonable en vez de por una derrota.

**3. Un intervalo de confianza cambia la conclusión, no la adorna.** La
caída de la U-Net entre prueba y Bolivia *parece* el resultado principal
hasta que se mide su intervalo y resulta indistinguible del ruido. Lo que
de verdad pasa es otra cosa.